# Sistema Inteligente de Control de Peaje — Inferencia Unificada (M1 + M2 + M3 + M4)

Este notebook integra **todos los modelos de visión por computadora en un solo lugar** y levanta el endpoint HTTPS (`INFERENCE_URL`) que consume la aplicación web PHP y la aplicación móvil:

| Modelo | Tarea | Implementación |
|---|---|---|
| **M1** | Detección de placas + OCR | `yolo-v9-t-384-license-plates-end2end.onnx` + `cct_xs_v2_global.onnx` (normalización boliviana `^\d{3,4}[A-Z]{3}$` + corrección `O↔0, I↔1, B↔8, S↔5`) |
| **M2** | Segmentación y tipo de vehículo (`motocicleta`, `automovil`, `minibus`, `camion`, `bus`) | Peso propio `m2_seg.pt` (YOLO11-seg) o fallback automático `YOLOE-11s-seg` |
| **M3** | Keypoints de ruedas → conteo de ejes (lado cercano) | Peso propio `m3_pose.pt` (YOLO11-pose 5 kps) o fallback automático `YOLOE-11s-seg` + keypoints geométricos |
| **M4** | *(Bono +5)* Conteo de pasajeros dentro del vehículo | `m4_personas.pt` o `yolo11n-seg.pt` (clase `person` intersectada con máscara M2; **sin rostros ni recortes**) |

> **Bonificación `venv` (+5 pts):** Todas las dependencias se instalan en un entorno virtual **`venv`** aislado (sin conda/mamba) y se guarda el diagnóstico del entorno en `evidencias_entornos/`.


## 1. Verificación de GPU y preparación del proyecto (Clonar o usar Drive)


In [ ]:
!nvidia-smi -L || echo "Sin GPU detectada: en Colab activa Entorno de ejecucion > Cambiar tipo de entorno > GPU T4"

import os
from pathlib import Path

# === CONFIGURACION ===
# Opcion A: URL de tu repositorio Git remoto (recomendado en Colab y RunPod)
REPO_URL = ""  # Ej: "https://github.com/tu-usuario/reconocimiento-peaje.git"
CARPETA_TRABAJO = Path("/content/reconocimiento-peaje")

# Opcion B: Si tus pesos entrenados (m2_seg.pt, m3_pose.pt, m4_personas.pt) estan en Google Drive:
USAR_DRIVE = False
CARPETA_PESOS_DRIVE = Path("/content/drive/MyDrive/peaje_modelos")

# Clave secreta que tambien pegaras en INFERENCE_API_KEY de sistema-peaje/.env
INFERENCE_API_KEY = "peaje-clave-segura-cambiar-2026"
# 1 = usar YOLOE/YOLO11-seg automaticamente si aun no tienes m2_seg.pt / m3_pose.pt
USE_ZEROSHOT_FALLBACK = "1"

if REPO_URL and not CARPETA_TRABAJO.exists():
    !git clone --depth 1 "{REPO_URL}" "{CARPETA_TRABAJO}"

DIR_INF = CARPETA_TRABAJO / "inferencia" if (CARPETA_TRABAJO / "inferencia").exists() else Path("/content/inferencia")
DIR_INF.mkdir(parents=True, exist_ok=True)

# Opcion C (pruebas rapidas sin GitHub): si falta app/main.py, sube inferencia_colab.zip desde tu PC
if not (DIR_INF / "app" / "main.py").is_file():
    import zipfile
    try:
        from google.colab import files
        print("No hay REPO_URL configurado. Sube el archivo 'inferencia_colab.zip' desde tu PC:")
        sub = files.upload()
        for nombre, contenido in sub.items():
            if nombre.endswith(".zip"):
                zpath = Path("/content") / nombre
                zpath.write_bytes(contenido)
                with zipfile.ZipFile(zpath, "r") as zf:
                    zf.extractall(DIR_INF)
                print("Extraido en:", DIR_INF)
    except ImportError:
        print("Copia la carpeta inferencia/ en:", DIR_INF)

print("Directorio de inferencia:", DIR_INF, "| app/main.py listo:", (DIR_INF / "app" / "main.py").is_file())


## 2. Creación del entorno `venv` e instalación reproducible (Bonificación +5 pts)


In [ ]:
import subprocess, sys
from pathlib import Path

VENV_DIR = DIR_INF / "venv"
VENV_PY = VENV_DIR / "bin" / "python"
VENV_PIP = VENV_DIR / "bin" / "pip"

if not VENV_PY.exists():
    py_ver = f"{sys.version_info.major}.{sys.version_info.minor}"
    subprocess.run(["bash", "-c", f"apt-get -qq update && apt-get -qq install -y python{py_ver}-venv python3-venv >/dev/null 2>&1"], check=False)
    r = subprocess.run([sys.executable, "-m", "venv", str(VENV_DIR)])
    if r.returncode != 0:
        # Fallback si ensurepip no esta disponible en el contenedor
        subprocess.run([sys.executable, "-m", "venv", "--without-pip", str(VENV_DIR)], check=True)
        subprocess.run(["bash", "-c", f"curl -sS https://bootstrap.pypa.io/get-pip.py | '{VENV_PY}'"], check=True)

subprocess.run([str(VENV_PIP), "install", "--upgrade", "pip"], check=True)
req = DIR_INF / "requirements-modelos.txt"
if req.is_file():
    subprocess.run([str(VENV_PIP), "install", "-r", str(req)], check=True)
else:
    subprocess.run([
        str(VENV_PIP), "install",
        "fastapi>=0.115.0", "uvicorn[standard]>=0.34.0", "python-multipart>=0.0.20",
        "onnxruntime>=1.21.0", "opencv-python-headless>=4.11.0", "numpy>=2.2.0",
        "PyYAML>=6.0.2", "requests>=2.32.0", "ultralytics>=8.3.0", "torch>=2.5.0", "torchvision>=0.20.0",
    ], check=True)

# Codificador CLIP para YOLOE (opcional para modo zero-shot)
subprocess.run([str(VENV_PIP), "install", "git+https://github.com/ultralytics/CLIP.git"], check=False)

# Guardar freeze exacto del entorno en la nube (evidencia bono +5 venv)
evid = DIR_INF / "evidencias_entornos"
evid.mkdir(parents=True, exist_ok=True)
frozen = subprocess.check_output([str(VENV_PIP), "freeze"], text=True)
(evid / "requirements_nube_frozen.txt").write_text(frozen, encoding="utf-8")
print("Entorno venv listo y dependencias instaladas:", VENV_PY)


## 3. Descarga verificada de M1 (SHA-256), carga de pesos propios y Diagnóstico del entorno


In [ ]:
import shutil

DIR_MODELOS = DIR_INF / "modelos"
DIR_MODELOS.mkdir(parents=True, exist_ok=True)

# Si USAR_DRIVE=True y existen en Drive, los copia; si ya estan en el repo o los subiste a /content, tambien los detecta
if USAR_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        for peso in ("m2_seg.pt", "m3_pose.pt", "m4_personas.pt"):
            src = CARPETA_PESOS_DRIVE / peso
            if src.is_file():
                shutil.copy2(src, DIR_MODELOS / peso)
                print("Copiado desde Drive:", peso)
    except Exception as e:
        print("Drive omitido:", e)

# Si el usuario subio m2_seg.pt o m3_pose.pt a la raiz /content/, moverlos a DIR_INF/modelos/
for peso in ("m2_seg.pt", "m3_pose.pt", "m4_personas.pt"):
    en_raiz = Path("/content") / peso
    if en_raiz.is_file() and not (DIR_MODELOS / peso).is_file():
        shutil.copy2(en_raiz, DIR_MODELOS / peso)
        print("Copiado desde /content:", peso)

# Descargar M1 (YOLOv9-t placas + fast-plate-ocr) verificando SHA-256
if (DIR_INF / "scripts" / "descargar_modelos.py").is_file():
    !{VENV_PY} "{DIR_INF / 'scripts' / 'descargar_modelos.py'}"
else:
    print("AVISO: Clona primero el repositorio (REPO_URL) o sube la carpeta inferencia/.")

print("\n=== ESTADO DE PESOS EN", DIR_MODELOS, "===")
for p in ("yolo-v9-t-384-license-plates-end2end.onnx", "cct_xs_v2_global.onnx", "m2_seg.pt", "m3_pose.pt"):
    ruta = DIR_MODELOS / p
    print(f" - {p}: {'LISTO (' + str(round(ruta.stat().st_size / 1e6, 1)) + ' MB)' if ruta.is_file() else 'NO ENCONTRADO (usara fallback)'}")

# Evidencia obligatoria para la bonificacion venv (+5 pts)
EVID_DIR = DIR_INF / "evidencias_entornos"
EVID_DIR.mkdir(parents=True, exist_ok=True)
if (DIR_INF / "scripts" / "diagnostico_entorno.py").is_file():
    !{VENV_PY} "{DIR_INF / 'scripts' / 'diagnostico_entorno.py'}" --salida "{EVID_DIR / 'diagnostico_nube.txt'}"
    !{VENV_PY} "{DIR_INF / 'scripts' / 'diagnostico_entorno.py'}" --json --salida "{EVID_DIR / 'diagnostico_nube.json'}"


## 4. Prueba visual unificada: M1 + M2 + M3 + M4 sobre una imagen

Ejecuta los **4 modelos juntos** dentro del `venv` sobre una foto (propia o de ejemplo), muestra el JSON exacto del contrato y genera una visualización con:
- Caja verde + texto de la placa (M1)
- Polígono/caja del vehículo + categoría (M2)
- Puntos de ruedas y conteo de ejes (M3)
- Conteo de pasajeros en cabecera (M4 — **nunca** guarda recortes de rostros)


In [ ]:
import json, urllib.request
from pathlib import Path
from IPython.display import Image, display

IMG_PRUEBA = DIR_INF / "prueba_vehiculo.jpg"
# Si estas en Colab y quieres subir una foto tuya, descomenta estas 3 lineas:
# from google.colab import files
# sub = files.upload()
# IMG_PRUEBA.write_bytes(next(iter(sub.values())))

if not IMG_PRUEBA.exists():
    urllib.request.urlretrieve("https://ultralytics.com/images/bus.jpg", IMG_PRUEBA)

OUT_IMG = DIR_INF / "resultado_unificado.jpg"
!USE_ZEROSHOT_FALLBACK={USE_ZEROSHOT_FALLBACK} {VENV_PY} "{DIR_INF / 'scripts' / 'demo_unificada.py'}" "{IMG_PRUEBA}" "{OUT_IMG}"
if OUT_IMG.exists():
    display(Image(filename=str(OUT_IMG)))


## 5. Levantar el Endpoint Único de Inferencia (FastAPI en el puerto 8000)

Esta celda arranca `uvicorn app.main:app` dentro del `venv` y lo expone con una URL HTTPS pública para pegarla en **`INFERENCE_URL`** dentro de `sistema-peaje/.env`:
- En **Google Colab**: descarga `cloudflared` en la VM de Colab (sin cuenta ni tarjeta) e imprime `https://xxxx.trycloudflare.com`.
- En **RunPod (+5 pts)**: usa la URL nativa del Pod `https://<RUNPOD_POD_ID>-8000.proxy.runpod.net`.


In [ ]:
import os, re, subprocess, time, requests

os.environ["INFERENCE_API_KEY"] = INFERENCE_API_KEY
os.environ["USE_ZEROSHOT_FALLBACK"] = USE_ZEROSHOT_FALLBACK

# Detener instancias previas si existieran
!pkill -f "uvicorn app.main:app" || true
!pkill -f "cloudflared tunnel" || true

log_api = open(DIR_INF / "api.log", "w")
proc_api = subprocess.Popen(
    [str(VENV_PY), "-m", "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd=str(DIR_INF), stdout=log_api, stderr=subprocess.STDOUT, env=os.environ.copy()
)

# Esperar a que /salud responda (la primera vez puede tardar 30-60s cargando pesos)
ok = False
for _ in range(90):
    try:
        r = requests.get("http://127.0.0.1:8000/salud", timeout=2)
        if r.ok:
            print("API lista:", r.json())
            ok = True
            break
    except Exception:
        pass
    time.sleep(2)

if not ok:
    print("ERROR: La API no respondio. Ultimas lineas de api.log:")
    print((DIR_INF / "api.log").read_text()[-2000:])
else:
    if os.environ.get("RUNPOD_POD_ID"):
        PUBLIC_URL = f"https://{os.environ['RUNPOD_POD_ID']}-8000.proxy.runpod.net"
        print("\n=== COPIA ESTO EN sistema-peaje/.env (RunPod) ===")
        print(f"INFERENCE_URL={PUBLIC_URL}")
        print(f"INFERENCE_API_KEY={INFERENCE_API_KEY}")
    else:
        # Exponer desde la VM de Colab mediante cloudflared
        !wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
        !chmod +x /usr/local/bin/cloudflared
        log_cf = open(DIR_INF / "tunnel.log", "w")
        subprocess.Popen(
            ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
            stdout=log_cf, stderr=subprocess.STDOUT
        )
        PUBLIC_URL = ""
        for _ in range(30):
            txt = (DIR_INF / "tunnel.log").read_text(errors="ignore")
            m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", txt)
            if m:
                PUBLIC_URL = m.group(0)
                break
            time.sleep(1)
        print("\n=== COPIA ESTO EN sistema-peaje/.env (Colab) ===")
        print(f"INFERENCE_URL={PUBLIC_URL}")
        print(f"INFERENCE_API_KEY={INFERENCE_API_KEY}")


## 6. Prueba en vivo contra el endpoint público (`GET /salud` y `POST /inferir`)


In [ ]:
import json, requests

assert PUBLIC_URL, "Ejecuta primero la celda anterior para obtener PUBLIC_URL."
print("GET /salud ->", requests.get(f"{PUBLIC_URL}/salud", timeout=15).json())

with open(IMG_PRUEBA, "rb") as f:
    r = requests.post(
        f"{PUBLIC_URL}/inferir",
        headers={"X-API-Key": INFERENCE_API_KEY},
        files={"imagen": ("vehiculo.jpg", f, "image/jpeg")},
        timeout=60,
    )
print("HTTP", r.status_code)
print(json.dumps(r.json(), indent=2, ensure_ascii=False))


## 7. Detener el servidor (cambiar `DETENER = True` solo al terminar)


In [ ]:
DETENER = False
if DETENER:
    !pkill -f "uvicorn app.main:app" || true
    !pkill -f "cloudflared tunnel" || true
    print("Servidor detenido.")
else:
    print("El servidor sigue activo en:", PUBLIC_URL)
